# Notebook 1 — Data validation & extraction for final BERTopic model (taxonomy + Radway)

This notebook is the **entry point** for Stage 10 analysis. It:
- loads the final BERTopic model with taxonomy & Radway mappings
- merges Stage 08 label metadata (label, scene summary, category tags)
- exports a **topic-level lookup table** used downstream to aggregate book-level proportions
- runs **QA checks** (missing mappings, keyword quality, confidence distribution)

Outputs are written to: `results/stage10_correlation_analysis/taxonomy_radway_eda/`


## 1) Setup & paths (keep consistent with your existing notebook)
This reproduces the same path conventions used in `radway_model_interactive_eda.ipynb`.

In [1]:
# --- Robust project_root fix: handle not defined and relative/invalid cases safely ---
from pathlib import Path

def safe_project_root(project_root_var=None) -> Path:
    """Ensure project_root is defined, exists, and is absolute.
    Attempts to infer sensible location if not provided or not valid."""
    # 1. Use arg if present, else try global, else fallback to cwd scan
    _pr = project_root_var
    try:
        if _pr is None:
            _pr = globals().get("project_root", None)
        # If still None or empty, try environment
        if _pr in (None, ""):
            _pr = Path.home()  # fallback to home, for now
        else:
            _pr = Path(_pr)
    except Exception:
        _pr = Path.cwd()
    
    _pr = _pr.expanduser().resolve()
    # If not valid, try scanning for src/results
    SEARCH_MARKERS = ["src", "results"]
    def looks_like_root(p):
        return all((p/pth).exists() for pth in SEARCH_MARKERS)
    
    # If path is ".", doesn't exist, or doesn't have src/results, look up tree
    if not _pr.exists() or _pr == Path(".") or not looks_like_root(_pr):
        # 1. Check up from cwd
        for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
            if looks_like_root(p):
                _pr = p
                break
        else:
            # 2. Fallback to hardcoded known path (edit if needed)
            KNOWN = Path("/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor")
            if looks_like_root(KNOWN):
                _pr = KNOWN.resolve()
            else:
                raise RuntimeError("Could not determine a valid project_root!")
    return _pr

project_root = safe_project_root()
print(f"✓ Project root: {project_root}")

✓ Project root: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor


In [2]:
from __future__ import annotations

import os
import json
import ast
from pathlib import Path
from typing import Any, Dict, Optional

import numpy as np
import pandas as pd

# Optional: model loading (only needed if you run this inside the project repo)
try:
    from bertopic import BERTopic
except Exception:
    BERTopic = None  # type: ignore

# ---- Project root detection ----
# Use project_root from previous cell if available, otherwise find it
if 'project_root' not in globals() or project_root is None:
    def find_project_root(start: Path | None = None) -> Path:
        """Find repo root by scanning parent dirs for `src/` (and optionally `results/`)."""
        start = (start or Path.cwd()).resolve()
        for p in [start, *start.parents]:
            if (p / "src").exists():
                return p
        return start
    
    project_root = Path(os.environ.get("PROJECT_ROOT", "")).expanduser()
    project_root = project_root if project_root.exists() else find_project_root()

print(f"Project root: {project_root}")

# ---- Load defaults from your project (if available) ----
try:
    from src.stage06_topic_exploration.explore_retrained_model import (
        DEFAULT_BASE_DIR,
        DEFAULT_EMBEDDING_MODEL,
    )
except Exception:
    # Fallbacks (edit if running outside the repo)
    DEFAULT_BASE_DIR = project_root / "models"
    DEFAULT_EMBEDDING_MODEL = "paraphrase-MiniLM-L6-v2"

print(f"DEFAULT_BASE_DIR: {DEFAULT_BASE_DIR}")
print(f"DEFAULT_EMBEDDING_MODEL: {DEFAULT_EMBEDDING_MODEL}")

# ---- Paths (match your existing notebook) ----
stage_subfolder = "stage09_category_mapping"
model_suffix = "_with_radway_mappings"

MODEL_PATH = DEFAULT_BASE_DIR / DEFAULT_EMBEDDING_MODEL / stage_subfolder / f"model_1{model_suffix}"

LABELS_FILENAME = (
    "labels_pos_openrouter_mistralai_Mistral-Nemo-Instruct-2407_"  # adjust if needed
    "romance_aware_paraphrase-MiniLM-L6-v2.json"
)
LABELS_PATH = project_root / "results" / "stage08_llm_labeling" / LABELS_FILENAME

OUT_DIR = project_root / "results" / "stage10_correlation_analysis" / "taxonomy_radway_eda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MODEL_PATH:  {MODEL_PATH}")
print(f"LABELS_PATH: {LABELS_PATH}")
print(f"OUT_DIR:     {OUT_DIR}")


Project root: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor
DEFAULT_BASE_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/models
DEFAULT_EMBEDDING_MODEL: paraphrase-MiniLM-L6-v2
MODEL_PATH:  /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/models/paraphrase-MiniLM-L6-v2/stage09_category_mapping/model_1_with_radway_mappings
LABELS_PATH: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage08_llm_labeling/labels_pos_openrouter_mistralai_Mistral-Nemo-Instruct-2407_romance_aware_paraphrase-MiniLM-L6-v2.json
OUT_DIR:     /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda


## 2) Load Stage 08 labels JSON (topic label + scene summary + tags)
This is the preferred source for `label`, `scene_summary`, and category tags.

In [3]:
def load_labels_metadata(labels_path: Path) -> dict[int, dict[str, Any]]:
    """Load full metadata from Stage 08 labels JSON.
    Accepts both:
      - rich JSON per topic (dict with label, scene_summary, etc.)
      - simple mapping {topic_id: "label"}
    """
    # Convert to Path if needed
    labels_path = Path(labels_path)
    
    # If path doesn't exist, try to find project root and reconstruct path
    if not labels_path.exists():
        # Find project root by looking for src/ and results/ directories
        from pathlib import Path as P
        cwd = P.cwd()
        project_root = None
        for parent in [cwd, *cwd.parents]:
            if (parent / "src").exists() and (parent / "results").exists():
                project_root = parent.resolve()
                break
        
        if project_root:
            # Extract the path components after project root
            # The labels_path should be: project_root / "results" / "stage08_llm_labeling" / filename
            path_str = str(labels_path)
            if "results" in path_str and "stage08_llm_labeling" in path_str:
                # Reconstruct: project_root / results / stage08_llm_labeling / filename
                filename = labels_path.name
                labels_path = project_root / "results" / "stage08_llm_labeling" / filename
    
    if not labels_path.exists():
        # Fall back: try to find any labels_*.json in the same folder
        labels_dir = labels_path.parent
        
        if not labels_dir.exists():
            raise FileNotFoundError(
                f"Labels directory does not exist: {labels_dir}"
            )
        
        # Try non-recursive glob first (files directly in the directory)
        candidates = sorted(labels_dir.glob("labels_*.json"))
        
        # If no files found, try recursive search (including subdirectories)
        if not candidates:
            candidates = sorted(labels_dir.rglob("labels_*.json"))
        
        if not candidates:
            # Provide helpful error message with available files
            available_files = [f.name for f in labels_dir.iterdir() if f.is_file()]
            raise FileNotFoundError(
                f"No labels file found at {labels_path}\n"
                f"Directory exists: {labels_dir}\n"
                f"Available files (first 10): {available_files[:10]}"
            )
        
        # Use the newest file (last in sorted list)
        labels_path = candidates[-1]
        print(f"⚠️ LABELS_PATH not found; using newest candidate: {labels_path.name}")

    print(f"Loading labels metadata from: {labels_path}")
    with open(labels_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    metadata: dict[int, dict[str, Any]] = {}
    for topic_id_str, topic_data in data.items():
        topic_id = int(topic_id_str)
        if isinstance(topic_data, dict):
            metadata[topic_id] = topic_data.copy()
        else:
            metadata[topic_id] = {"label": str(topic_data)}

    print(f"✓ Loaded Stage 08 label metadata for {len(metadata)} topics")
    return metadata

labels_metadata = load_labels_metadata(LABELS_PATH)
list(labels_metadata.items())[:2]


Loading labels metadata from: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage08_llm_labeling/labels_pos_openrouter_mistralai_Mistral-Nemo-Instruct-2407_romance_aware_paraphrase-MiniLM-L6-v2.json
✓ Loaded Stage 08 label metadata for 361 topics


[(0,
  {'label': 'Negotiating Deal',
   'keywords': ['means',
    'idea',
    'promise',
    'work',
    'help',
    'better',
    'true',
    'deal',
    'today',
    'game'],
   'primary_categories': ['relationship_conflict', 'domestic_life'],
   'secondary_categories': ['setting:living_room', 'activity:discussion'],
   'is_noise': False,
   'rationale': "The top keywords 'means', 'deal', 'today' and 'work' suggest a negotiation. The snippets show a conversation between two people, indicating a domestic life scene with relationship conflict.",
   'scene_summary': 'The couple discusses terms and expectations for their relationship.'}),
 (1,
  {'label': 'Intimate Breast And Nipple Kissing',
   'keywords': ['hips',
    'tongue',
    'breasts',
    'mouth',
    'body',
    'arms',
    'neck',
    'thighs',
    'nipples',
    'hand'],
   'primary_categories': ['physical_affection', 'sexual_content'],
   'secondary_categories': ['setting:bedroom', 'activity:kissing'],
   'is_noise': False,

## 3) Load model (or fall back to previously exported `full_model_data.csv`)
If you run this inside the repo and the model exists, it loads the BERTopic model.
If not, it loads the last exported topic table from `OUT_DIR/full_model_data.csv`.

In [4]:
def load_model(model_path: Path):
    if BERTopic is None:
        raise ImportError("BERTopic is not available in this environment.")
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found at: {model_path}")
    return BERTopic.load(str(model_path))

# Fix: Ensure OUT_DIR and all path variables match the expected absolute paths.
FULL_CSV = OUT_DIR / "full_model_data.csv"
ARCHIVE_CSV = OUT_DIR / "archive" / "full_model_data.csv"
ARCHIVE_PARQUET = OUT_DIR / "archive" / "full_model_data.parquet"

# Print the resolved absolute paths to avoid confusion
print(f"FULL_CSV: {FULL_CSV.resolve()}")
print(f"ARCHIVE_CSV: {ARCHIVE_CSV.resolve()}")
print(f"ARCHIVE_PARQUET: {ARCHIVE_PARQUET.resolve()}")
print(f"MODEL_PATH: {MODEL_PATH.resolve()}")

# Additionally check alternate model directory (if present)
RETRAINED_MODEL_PATH = Path(
    "/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/models/retrained/paraphrase-MiniLM-L6-v2/stage09_category_mapping/model_1_with_radway_mappings"
)
model = None

tried_model_paths = [MODEL_PATH]
if RETRAINED_MODEL_PATH.exists():
    tried_model_paths.insert(0, RETRAINED_MODEL_PATH)

model_loaded = False
for candidate_model_path in tried_model_paths:
    if candidate_model_path.exists() and BERTopic is not None:
        print(f"🔍 Trying to load BERTopic model from: {candidate_model_path}")
        model = load_model(candidate_model_path)
        print(f"✓ Model loaded from: {candidate_model_path}")
        model_loaded = True
        break

if not model_loaded:
    print("⚠️ Model not available at the checked locations; will use exported CSV if present.")
    # Print which files exist for debugging
    for file_path in [FULL_CSV, ARCHIVE_CSV, ARCHIVE_PARQUET]:
        print(f"Exists ({file_path}): {file_path.exists()}")

    if not FULL_CSV.exists() and not ARCHIVE_CSV.exists() and not ARCHIVE_PARQUET.exists():
        print(f"   Expected: {FULL_CSV.resolve()}")
        print(f"        or: {ARCHIVE_CSV.resolve()}")
        print(f"        or: {ARCHIVE_PARQUET.resolve()}")

FULL_CSV: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda/full_model_data.csv
ARCHIVE_CSV: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda/archive/full_model_data.csv
ARCHIVE_PARQUET: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda/archive/full_model_data.parquet
MODEL_PATH: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/models/paraphrase-MiniLM-L6-v2/stage09_category_mapping/model_1_with_radway_mappings
🔍 Trying to load BERTopic model from: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/models/retrained/paraphrase-MiniLM-L6-v2/stage09_category_mapping/model_1_with_radway_mappings
✓ Model loaded

## 4) Extract topic-level table
This table is the *topic lookup* you will later join with `book_topic_probs.parquet` and `chapter_topic_probs.parquet` (outputs from `generate_topic_probabilities_goodreads.py` in `results/stage10_correlation_analysis/`).

In [5]:
def extract_all_fields(
    model,
    labels_metadata: Optional[dict[int, dict[str, Any]]] = None,
) -> pd.DataFrame:
    rows = []

    # topic ids (exclude -1 outlier)
    topic_ids = [tid for tid in getattr(model, "topic_representations_", {}).keys() if tid != -1]

    for topic_id in sorted(topic_ids):
        row: dict[str, Any] = {"topic_id": topic_id}

        # Keywords
        if hasattr(model, "topic_representations_") and topic_id in model.topic_representations_:
            kws = model.topic_representations_[topic_id]
            row["keywords"] = ", ".join([kw[0] for kw in kws[:10]])
            row["num_keywords"] = len(kws)
            row["all_keywords"] = [kw[0] for kw in kws]

        # Stage 08 label metadata (preferred)
        if labels_metadata and topic_id in labels_metadata:
            meta = labels_metadata[topic_id]
            row["label"] = meta.get("label")
            row["scene_summary"] = meta.get("scene_summary")
            row["primary_categories"] = ", ".join(meta.get("primary_categories", [])) or None
            row["secondary_categories"] = ", ".join(meta.get("secondary_categories", [])) or None
            row["label_is_noise"] = bool(meta.get("is_noise", False))
            row["label_rationale"] = meta.get("rationale")
        else:
            row["label"] = getattr(model, "topic_labels_", {}).get(topic_id)
            row["scene_summary"] = None
            row["primary_categories"] = None
            row["secondary_categories"] = None
            row["label_is_noise"] = None
            row["label_rationale"] = None

        # Taxonomy (Stage 2)
        tax = getattr(model, "topic_taxonomy_", {}).get(topic_id, {})
        row.update({
            "taxonomy_main_id": tax.get("main_category_id"),
            "taxonomy_main_name": tax.get("main_category_name"),
            "taxonomy_main_group": tax.get("main_category_group"),
            "taxonomy_secondary_id": tax.get("secondary_category_id"),
            "taxonomy_secondary_name": tax.get("secondary_category_name"),
            "taxonomy_secondary_group": tax.get("secondary_category_group"),
            "taxonomy_confidence": tax.get("confidence"),
            "taxonomy_is_noise": bool(tax.get("is_noise", False)),
        })

        # Radway (Stage 3)
        rad = getattr(model, "topic_radway_", {}).get(topic_id, {})
        row.update({
            "radway_main_id": rad.get("radway_main_id"),
            "radway_main_name": rad.get("radway_main_name"),
            "radway_secondary_id": rad.get("radway_secondary_id"),
            "radway_phase": rad.get("radway_phase") if rad.get("radway_phase") is not None else "NA",
            "radway_phase_name": rad.get("radway_phase_name"),
            "radway_is_none": bool(rad.get("radway_is_none", False)),
            "radway_confidence": rad.get("radway_confidence"),
            "radway_rationale": rad.get("radway_rationale"),
        })

        rows.append(row)

    return pd.DataFrame(rows)

if model is not None:
    df_topics = extract_all_fields(model, labels_metadata=labels_metadata)
else:
    # fallback: load exported
    df_topics = pd.read_csv(FULL_CSV)
    print(f"✓ Loaded exported table: {FULL_CSV} ({df_topics.shape[0]} rows)")

df_topics.shape


(368, 26)

## 5) QA checks (Notebook 1 should keep these)
These checks prevent silent downstream errors when you aggregate to book-level.

In [6]:
# Basic schema sanity
expected_cols = {
    "topic_id","label","taxonomy_main_id","taxonomy_main_name","taxonomy_main_group",
    "radway_main_id","radway_main_name","radway_phase_name","radway_is_none"
}
missing = expected_cols - set(df_topics.columns)
if missing:
    raise ValueError(f"Missing expected columns: {sorted(missing)}")

# Duplicates
dup = df_topics["topic_id"].duplicated().sum()
print(f"Duplicate topic_id rows: {dup}")

# Missing mappings
n_total = len(df_topics)
n_tax_missing = df_topics["taxonomy_main_id"].isna().sum()
n_rad_missing = df_topics["radway_main_id"].isna().sum()
print(f"Taxonomy missing: {n_tax_missing}/{n_total}")
print(f"Radway missing:   {n_rad_missing}/{n_total}")

# Radway none vs function
rad_none = (df_topics["radway_is_none"] == True).sum()
rad_fn = (df_topics["radway_is_none"] == False).sum()
print(f"Radway function topics: {rad_fn}")
print(f"Radway none topics:     {rad_none}")

# Keyword-quality check: many empty tokens often indicate artifacts
def empty_ratio(all_keywords_cell) -> float:
    # all_keywords might be a list (fresh extract) or a string (CSV)
    if isinstance(all_keywords_cell, str):
        try:
            lst = ast.literal_eval(all_keywords_cell)
        except Exception:
            return np.nan
    else:
        lst = all_keywords_cell
    if not lst:
        return np.nan
    return sum(1 for w in lst if not w) / len(lst)

df_topics["_empty_kw_ratio"] = df_topics["all_keywords"].apply(empty_ratio)
bad_kw = df_topics[df_topics["_empty_kw_ratio"] > 0.5].copy()

print(f"Topics with >50% empty keywords: {len(bad_kw)}")
bad_kw[["topic_id","label","taxonomy_main_name","radway_main_id","_empty_kw_ratio"]].head(20)


Duplicate topic_id rows: 0
Taxonomy missing: 7/368
Radway missing:   7/368
Radway function topics: 272
Radway none topics:     96
Topics with >50% empty keywords: 16


,topic_id,label,taxonomy_main_name,radway_main_id,_empty_kw_ratio
17,17,17____,None,None,1.000000
18,18,18____,None,None,1.000000
120,120,Repeated Affirmations,"Bonding, Everyday Intimacy & Growth",R9,0.629630
132,132,132____,None,None,1.000000
141,141,Hurried Conversations,"Bonding, Everyday Intimacy & Growth",R5,0.962963
182,182,Cheek Heating Conversation,Positive Emotions & Security,none,0.629630
183,183,Division Discussion,"Bonding, Everyday Intimacy & Growth",R9,0.851852
186,186,186____,None,None,1.000000
205,205,Rule-breaking Encounter,Explicit Sexual Acts,R12,0.592593
224,224,224____,None,None,1.000000


In [7]:
# Save a 'needs review' list for manual inspection
needs_review = df_topics[
    df_topics["taxonomy_main_id"].isna()
    | df_topics["radway_main_id"].isna()
    | (df_topics["_empty_kw_ratio"] > 0.5)
][["topic_id","label","keywords","taxonomy_main_name","taxonomy_main_group","radway_main_id","radway_phase_name","taxonomy_confidence","radway_confidence","_empty_kw_ratio"]]

needs_review_path = OUT_DIR / "topics_needs_review.csv"
needs_review.to_csv(needs_review_path, index=False)
print(f"✓ Wrote: {needs_review_path}  (n={len(needs_review)})")


✓ Wrote: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda/topics_needs_review.csv  (n=16)


## 6) Export topic lookup + summary tables
These outputs are used downstream to build book-level category proportions and indices.

In [8]:
# Summary statistics (same fields as your prior notebook)
summary = {
    "total_topics": int(len(df_topics)),
    "topics_with_labels": int(df_topics["label"].notna().sum()),
    "topics_with_taxonomy": int(df_topics["taxonomy_main_id"].notna().sum()),
    "topics_with_radway": int(df_topics["radway_main_id"].notna().sum()),
    "topics_with_radway_function": int((df_topics["radway_is_none"] == False).sum()),
    "topics_with_radway_none": int((df_topics["radway_is_none"] == True).sum()),
    "unique_taxonomy_categories": int(df_topics["taxonomy_main_name"].nunique(dropna=True)),
    "unique_taxonomy_groups": int(df_topics["taxonomy_main_group"].nunique(dropna=True)),
    "unique_radway_functions": int(df_topics.loc[df_topics["radway_is_none"] == False, "radway_main_name"].nunique(dropna=True)),
    "unique_radway_phases": int(df_topics["radway_phase_name"].nunique(dropna=True)),
}
summary_path = OUT_DIR / "summary_statistics.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"✓ Wrote: {summary_path}")

        # Topic lookup (for merging with book_topic_probs.parquet later)
topic_lookup = df_topics[[
    "topic_id",
    "taxonomy_main_id","taxonomy_main_name","taxonomy_main_group",
    "taxonomy_secondary_id","taxonomy_secondary_name","taxonomy_secondary_group",
    "taxonomy_confidence","taxonomy_is_noise",
    "radway_main_id","radway_main_name","radway_phase","radway_phase_name","radway_is_none","radway_confidence",
    "label","scene_summary","primary_categories","secondary_categories","label_is_noise",
]].copy()

topic_lookup_path = OUT_DIR / "topic_lookup.parquet"
topic_lookup.to_parquet(topic_lookup_path, index=False)
print(f"✓ Wrote: {topic_lookup_path}")

# Also export full table (CSV + Parquet) to keep parity with your existing notebook
full_csv = OUT_DIR / "full_model_data.csv"
full_parquet = OUT_DIR / "full_model_data.parquet"
df_topics.to_csv(full_csv, index=False)
df_topics.to_parquet(full_parquet, index=False)
print(f"✓ Wrote: {full_csv}")
print(f"✓ Wrote: {full_parquet}")

# Cross-tabs to reuse in EDA notebook (Notebook 3), but stored here for convenience
ct_group_phase = pd.crosstab(df_topics["taxonomy_main_group"], df_topics["radway_phase_name"])
ct_group_phase.to_csv(OUT_DIR / "crosstab_taxonomy_group_x_radway_phase.csv")

ct_top = pd.crosstab(df_topics["taxonomy_main_name"], df_topics["radway_main_id"])
ct_top.to_csv(OUT_DIR / "crosstab_taxonomy_main_x_radway_id.csv")

print("✓ Wrote crosstab CSVs")


✓ Wrote: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda/summary_statistics.json
✓ Wrote: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda/topic_lookup.parquet
✓ Wrote: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda/full_model_data.csv
✓ Wrote: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/taxonomy_radway_eda/full_model_data.parquet
✓ Wrote crosstab CSVs


## What comes next
- **Notebook 2** will merge `topic_lookup.parquet` with `book_topic_probs.parquet` / `chapter_topic_probs.parquet` (generated by `generate_topic_probabilities_goodreads.py` in `results/stage10_correlation_analysis/`) and produce book-level category proportions.
- **Notebook 3** will use those outputs for EDA (heatmaps, volcano plots, correlations, clustering).


## Book ID strategy (Goodreads-based) + join health checks

Your downstream statistical results depend on being able to **merge**:
- topic/category outputs (from `generate_topic_probabilities*.py` and the category-mapping stage)
- Goodreads metadata (ratings, shelves, etc.)
- chapter/book feature tables (if used)
- Radway overlays (if used)

To avoid silent failures (e.g., `overlap_books = 0`), this section enforces a **single canonical ID**:
**`book_id := goodreads_book_id`** (string, stripped), and runs overlap diagnostics before any hypothesis tests.

In [9]:
from pathlib import Path
import pandas as pd

def normalize_id(s: pd.Series) -> pd.Series:
    """Normalize identifiers for safe joins.
    
    This function handles the output format from generate_topic_probabilities_goodreads.py,
    which may produce IDs with .0 suffixes (e.g., '60416566.0') when parquet files store
    numeric IDs as floats. This normalizes them to match the format in goodreads.csv.
    
    Handles:
    - Float strings like '60416566.0' -> '60416566' (matches script output format)
    - Whitespace trimming
    - Empty/null values
    """
    s = s.astype("string").str.strip()
    # Remove .0 suffix from float strings (e.g., '60416566.0' -> '60416566')
    # This handles IDs from generate_topic_probabilities_goodreads.py output
    s = s.str.replace(r'\.0+$', '', regex=True)
    s = s.mask(s.str.lower().isin(["", "nan", "none"]))
    return s

def assert_overlap(a: pd.Series, b: pd.Series, name_a: str, name_b: str, min_share: float = 0.5):
    """Check overlap between two ID series and report coverage metrics.
    
    Returns:
        overlap: set of overlapping IDs
        share: overlap / min(len(A), len(B)) - useful when checking if outputs are subset of meta
        coverage_a: overlap / len(A) - coverage of A by B
        coverage_b: overlap / len(B) - coverage of B by A
    """
    A = set(a.dropna().unique().tolist())
    B = set(b.dropna().unique().tolist())
    overlap = A & B
    
    # Multiple metrics for clarity
    share = len(overlap) / max(1, min(len(A), len(B)))  # Overlap relative to smaller set
    coverage_a = len(overlap) / max(1, len(A))  # How much of A is covered by B
    coverage_b = len(overlap) / max(1, len(B))  # How much of B is covered by A
    
    print(f"{name_a}: {len(A):,} unique")
    print(f"{name_b}: {len(B):,} unique")
    print(f"OVERLAP: {len(overlap):,}")
    print(f"  Overlap/min: {share:.2%} (overlap relative to smaller set)")
    print(f"  Coverage of {name_a} by {name_b}: {coverage_a:.2%}")
    print(f"  Coverage of {name_b} by {name_a}: {coverage_b:.2%}")
    
    # Show sample of actual overlapping IDs (not just random samples from each set)
    if overlap:
        sample_overlap = sorted(list(overlap))[:5]
        print(f"  Sample overlapping IDs: {sample_overlap}")
    
    if len(overlap) == 0:
        # Show sample values to help debug
        sample_a = sorted(list(A))[:5] if A else []
        sample_b = sorted(list(B))[:5] if B else []
        raise AssertionError(
            f"❌ Zero overlap between {name_a} and {name_b}.\n"
            f"   Sample {name_a} IDs: {sample_a}\n"
            f"   Sample {name_b} IDs: {sample_b}\n"
            f"   Fix: Check that book_id values match between datasets (may need to check ID column names or merge upstream)."
        )
    if share < min_share:
        print(f"⚠️ Low overlap (<{min_share:.0%}). This may still be a problem—inspect mismatched IDs.")
    return overlap, share, coverage_a, coverage_b

# ---- Configure likely locations (edit if your repo differs) ----
PROJECT_ROOT = globals().get("project_root", Path.cwd()).resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

GOODREADS_PATH_CANDIDATES = [
    DATA_DIR / "goodreads.csv",
    DATA_DIR / "books_meta.csv",
    RESULTS_DIR / "books_meta.csv",
]

# Find Goodreads/meta file
goodreads_path = next((p for p in GOODREADS_PATH_CANDIDATES if p.exists()), None)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("goodreads/meta path:", goodreads_path)

if goodreads_path is None:
    print("⚠️ Could not find goodreads.csv / books_meta.csv automatically. Add its path above.")
else:
    meta = pd.read_csv(goodreads_path)
    # Choose Goodreads id column (check multiple variants)
    goodreads_id_col = None
    # Priority order: prefer explicit Goodreads ID columns, then common variants, then generic ID
    for c in ["goodreads_book_id", "book_id", "goodreads_id", "work_id", "ID", "id"]:
        if c in meta.columns:
            goodreads_id_col = c
            break
    if goodreads_id_col is None:
        raise ValueError(f"No Goodreads id column found in {goodreads_path}. Columns: {meta.columns.tolist()}")
    meta["book_id"] = normalize_id(meta[goodreads_id_col])
    print("Meta ID column:", goodreads_id_col, " | unique book_id:", meta["book_id"].nunique())

PROJECT_ROOT: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor
goodreads/meta path: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/data/processed/goodreads.csv
Meta ID column: ID  | unique book_id: 97


### Overlap diagnostics against generated outputs

This cell checks overlap between Goodreads metadata and generated outputs:
- `book_topic_probs.parquet` / `chapter_topic_probs.parquet` (from `generate_topic_probabilities_goodreads.py`)
- Category proportion tables (if available)

**Note**: If you see zero overlap, check:
1. The `book_id` column in `book_topic_probs.parquet` should match the `ID` column from `goodreads.csv` (after normalization)
2. The script `generate_topic_probabilities_goodreads.py` uses `--goodreads-id-col` to specify which column contains Goodreads IDs in the sentence dataframe
3. Ensure the sentence dataframe was merged with Goodreads metadata upstream

It will compute overlaps and write a small report CSV you can keep under `results/stage10_correlation_analysis/id_alignment_report.csv`.

In [10]:
import pandas as pd
from typing import Optional

# ---- Create mapping from chapters book_id to Goodreads ID ----
# The book_topic_probs may use book_id from chapters.csv, which needs mapping to Goodreads ID
def create_book_id_mapping(goodreads_path: Path, chapters_path: Optional[Path] = None) -> Optional[pd.DataFrame]:
    """Create a mapping from chapters book_id to Goodreads ID.
    
    Returns a DataFrame with columns: chapters_book_id, goodreads_id
    """
    # Try to find chapters.csv
    if chapters_path is None:
        chapters_candidates = [
            DATA_DIR / "chapters.csv",
            PROJECT_ROOT / "data" / "raw" / "chapters.csv",
            PROJECT_ROOT / "data" / "chapters.csv",
        ]
        chapters_path = next((p for p in chapters_candidates if p.exists()), None)
    
    if chapters_path is None or not chapters_path.exists():
        print("⚠️ Could not find chapters.csv to create book_id mapping")
        return None
    
    try:
        chapters_df = pd.read_csv(chapters_path)
        goodreads_df = pd.read_csv(goodreads_path)
        
        # Check what ID columns exist
        chapters_id_col = None
        for col in ["book_id", "ID", "id"]:
            if col in chapters_df.columns:
                chapters_id_col = col
                break
        
        goodreads_id_col = None
        for col in ["ID", "goodreads_book_id", "goodreads_id"]:
            if col in goodreads_df.columns:
                goodreads_id_col = col
                break
        
        if chapters_id_col is None or goodreads_id_col is None:
            print(f"⚠️ Missing ID columns: chapters={chapters_id_col}, goodreads={goodreads_id_col}")
            return None
        
        # Try to match by Author + Title if direct ID match doesn't work
        if "Author" in chapters_df.columns and "Book Title" in chapters_df.columns:
            if "Author" in goodreads_df.columns and "Title" in goodreads_df.columns:
                # Merge on Author + Title
                mapping = chapters_df[[chapters_id_col, "Author", "Book Title"]].merge(
                    goodreads_df[[goodreads_id_col, "Author", "Title"]],
                    left_on=["Author", "Book Title"],
                    right_on=["Author", "Title"],
                    how="inner"
                )
                mapping = mapping[[chapters_id_col, goodreads_id_col]].rename(columns={
                    chapters_id_col: "chapters_book_id",
                    goodreads_id_col: "goodreads_id"
                })
                mapping["chapters_book_id"] = normalize_id(mapping["chapters_book_id"])
                mapping["goodreads_id"] = normalize_id(mapping["goodreads_id"])
                print(f"✓ Created mapping from {len(mapping)} matched books (Author+Title)")
                return mapping
        
        # Fallback: try direct ID match if they're the same format
        if chapters_id_col and goodreads_id_col:
            chapters_ids = normalize_id(chapters_df[chapters_id_col]).dropna().unique()
            goodreads_ids = normalize_id(goodreads_df[goodreads_id_col]).dropna().unique()
            overlap = set(chapters_ids) & set(goodreads_ids)
            if len(overlap) > 0:
                print(f"⚠️ Found {len(overlap)} overlapping IDs, but format may differ. Using Author+Title mapping instead.")
        
        return None
    except Exception as e:
        print(f"⚠️ Error creating book_id mapping: {e}")
        return None

# ---- Configure/locate common output files (edit if needed) ----
CANDIDATE_OUTPUTS = {
    # topic probabilities (produced by generate_topic_probabilities_goodreads.py)
    "book_topic_probs": [
        RESULTS_DIR / "stage10_correlation_analysis" / "book_topic_probs.parquet",  # Primary: generated output
        RESULTS_DIR / "stage10_correlation_analysis" / "book_topic_probs.csv",      # Fallback: CSV if exists
        RESULTS_DIR / "stage08_bertopic" / "book_topic_probs.csv",                 # Legacy location
    ],
    "chapter_topic_probs": [
        RESULTS_DIR / "stage10_correlation_analysis" / "chapter_topic_probs.parquet",  # Primary: generated output
        RESULTS_DIR / "stage10_correlation_analysis" / "chapter_topic_probs.csv",      # Fallback: CSV if exists
        RESULTS_DIR / "stage08_bertopic" / "chapter_topic_probs.csv",                 # Legacy location
    ],
    # category proportions (after mapping topics -> taxonomy)
    "book_category_props": [
        RESULTS_DIR / "stage09_category_mapping" / "stage2_theory_driven_categories" / "book_category_proportions.parquet",
        RESULTS_DIR / "stage10_correlation_analysis" / "book_category_props.csv",
    ],
}

def load_first_existing(candidates):
    for p in candidates:
        if p.exists():
            return p
    return None

loaded = {}
for name, cands in CANDIDATE_OUTPUTS.items():
    p = load_first_existing(cands)
    loaded[name] = p
    print(f"{name}: {p}")

# Create book_id mapping if needed
book_id_mapping = None
if goodreads_path is not None:
    book_id_mapping = create_book_id_mapping(goodreads_path)
    if book_id_mapping is not None:
        print(f"\n✓ Book ID mapping available: {len(book_id_mapping)} books")

# Load and compare overlaps
reports = []

if goodreads_path is not None:
    meta_ids = meta["book_id"]
    for name, p in loaded.items():
        if p is None:
            continue
        if p.suffix == ".parquet":
            df = pd.read_parquet(p)
        else:
            df = pd.read_csv(p)

        # Determine id column (check multiple variants)
        id_col = None
        # Priority: book_id (standardized), then Goodreads-specific columns, then generic ID
        for c in ["book_id", "goodreads_book_id", "goodreads_id", "work_id", "bookId", "ID", "id"]:
            if c in df.columns:
                id_col = c
                break
        if id_col is None:
            print(f"⚠️ {name}: no id column found. Columns: {df.columns.tolist()}")
            continue

        df_ids = normalize_id(df[id_col])
        
        # Note: normalize_id() already handles .0 suffix removal, so IDs should match now
        # If mapping is available and IDs still don't match, try mapping as fallback
        if book_id_mapping is not None:
            # Quick check: if still no overlap after normalization, try mapping
            sample_df_ids = set(df_ids.dropna().head(10).tolist())
            sample_meta_ids = set(meta_ids.dropna().head(10).tolist())
            
            if not (sample_df_ids & sample_meta_ids):
                print(f"  🔄 No overlap after normalization, attempting book_id mapping...")
                # Map chapters book_id to Goodreads ID
                df_mapped = pd.DataFrame({"chapters_book_id": df_ids})
                df_mapped = df_mapped.merge(
                    book_id_mapping,
                    left_on="chapters_book_id",
                    right_on="chapters_book_id",
                    how="left"
                )
                mapped_count = df_mapped["goodreads_id"].notna().sum()
                if mapped_count > 0:
                    print(f"  ✓ Mapped {mapped_count}/{len(df_mapped)} ({mapped_count/len(df_mapped)*100:.1f}%) IDs")
                    df_ids = df_mapped["goodreads_id"]  # Use mapped Goodreads IDs
                else:
                    print(f"  ⚠️ Mapping failed - no matches found")
        
        print(f"\n--- {name} vs goodreads/meta ---")
        overlap, share, coverage_outputs, coverage_meta = assert_overlap(df_ids, meta_ids, name, "goodreads/meta", min_share=0.5)
        reports.append({
            "table": name,
            "path": str(p),
            "id_col": id_col,
            "unique_ids": int(df_ids.nunique()),
            "meta_unique_ids": int(meta_ids.nunique()),
            "overlap_ids": int(len(overlap)),
            "overlap_share": float(share),
            "coverage_of_meta": float(coverage_meta),  # How much of metadata is covered by outputs
            "coverage_of_outputs": float(coverage_outputs),  # How much of outputs is covered by metadata
        })

    report_df = pd.DataFrame(reports).sort_values("overlap_share", ascending=False)
    print("\n--- overlap report ---")
    display(report_df)

    # Save report next to your stage10 outputs (safe default)
    OUT = RESULTS_DIR / "stage10_correlation_analysis" / "id_alignment_report.csv"
    OUT.parent.mkdir(parents=True, exist_ok=True)
    report_df.to_csv(OUT, index=False)
    print("✓ Wrote:", OUT)
    
    # Validation summary
    min_overlap_share = report_df["overlap_share"].min()
    min_coverage_meta = report_df["coverage_of_meta"].min()  # Coverage of metadata by outputs
    
    print("\n" + "="*60)
    print("VALIDATION SUMMARY")
    print("="*60)
    
    if min_coverage_meta >= 0.95:
        print(f"✅ SUCCESS: All datasets cover ≥95% of metadata (coverage: {min_coverage_meta:.1%})")
        print("   Ready for downstream analysis!")
    elif min_coverage_meta >= 0.90:
        print(f"⚠️  WARNING: Metadata coverage is <95% (coverage: {min_coverage_meta:.1%})")
        print(f"   {int((1-min_coverage_meta) * report_df['meta_unique_ids'].iloc[0])} books from metadata are missing in outputs.")
        print("   This is acceptable if those books have no usable text, but review missing books.")
    else:
        print(f"❌ ERROR: Low metadata coverage detected (coverage: {min_coverage_meta:.1%})")
        print(f"   {int((1-min_coverage_meta) * report_df['meta_unique_ids'].iloc[0])} books from metadata are missing in outputs.")
        print("   Review missing books before proceeding.")
    
    if min_overlap_share >= 0.95:
        print(f"\n✅ Internal consistency: All output datasets are internally consistent (overlap: {min_overlap_share:.1%})")
    else:
        print(f"\n⚠️  Internal consistency issue: Some datasets have mismatched IDs (overlap: {min_overlap_share:.1%})")
    
    print("\nNote: 'coverage_of_meta' shows how much of the metadata is represented in outputs.")
    print("      'overlap_share' shows internal consistency between output datasets.")

book_topic_probs: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/book_topic_probs.parquet
chapter_topic_probs: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/chapter_topic_probs.parquet
book_category_props: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage09_category_mapping/stage2_theory_driven_categories/book_category_proportions.parquet
⚠️ Missing ID columns: chapters=None, goodreads=ID

--- book_topic_probs vs goodreads/meta ---
book_topic_probs: 92 unique
goodreads/meta: 97 unique
OVERLAP: 92
  Overlap/min: 100.00% (overlap relative to smaller set)
  Coverage of book_topic_probs by goodreads/meta: 100.00%
  Coverage of goodreads/meta by book_topic_probs: 94.85%
  Sample overlapping IDs: ['104659050', '11266880', '123257687', '123446478', '127305713']

--- chapte

,table,path,id_col,unique_ids,meta_unique_ids,overlap_ids,overlap_share,coverage_of_meta,coverage_of_outputs
0,book_topic_probs,/home/polina/Documents/goodreads_romance_resea...,book_id,92,97,92,1.0,0.948454,1.0
1,chapter_topic_probs,/home/polina/Documents/goodreads_romance_resea...,book_id,92,97,92,1.0,0.948454,1.0
2,book_category_props,/home/polina/Documents/goodreads_romance_resea...,book_id,92,97,92,1.0,0.948454,1.0


✓ Wrote: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/id_alignment_report.csv

VALIDATION SUMMARY
⚠️  WARNING: Metadata coverage is <95% (coverage: 94.8%)
   5 books from metadata are missing in outputs.
   This is acceptable if those books have no usable text, but review missing books.

✅ Internal consistency: All output datasets are internally consistent (overlap: 100.0%)

Note: 'coverage_of_meta' shows how much of the metadata is represented in outputs.
      'overlap_share' shows internal consistency between output datasets.


### Diagnostic: Trace missing books through pipeline

This cell identifies which books from metadata are missing in outputs and traces where they were lost:
1. Missing in `chapters.csv` (never scraped/extracted)
2. Present in `chapters.csv` but missing in `sentence_df_with_topics.parquet` (filtered during preprocessing)
3. Present in `sentence_df` but missing in outputs (aggregation issue - less likely)


In [11]:
from pathlib import Path
import pandas as pd

def norm(s): 
    """Normalize ID series (same as normalize_id but for single series)."""
    s = s.astype("string").str.strip()
    s = s.str.replace(r'\.0+$', '', regex=True)
    s = s.mask(s.str.lower().isin(["", "nan", "none"]))
    return s

PROJECT_ROOT = globals().get("project_root", Path.cwd()).resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

chapters_path = DATA_DIR / "chapters.csv"
sent_path = DATA_DIR / "sentence_df_with_topics.parquet"

# --- Get meta ids (already normalized from previous cell) ---
meta_ids_normalized = meta["book_id"]  # Already normalized
meta_set = set(meta_ids_normalized.dropna().unique())

# --- Get output ids (pick book_topic_probs as representative) ---
book_topic_path = RESULTS_DIR / "stage10_correlation_analysis" / "book_topic_probs.parquet"
if book_topic_path.exists():
    book_topic = pd.read_parquet(book_topic_path)
    out_ids = norm(book_topic["book_id"])
    out_set = set(out_ids.dropna().unique())
    
    missing_from_outputs = sorted(meta_set - out_set)
    print(f"📊 Missing from outputs: {len(missing_from_outputs)} books")
    print(f"   Missing IDs: {missing_from_outputs}")
    
    if missing_from_outputs:
        missing_meta_rows = meta[meta["book_id"].isin(missing_from_outputs)].copy()
        print(f"\n📋 Metadata for missing books:")
        # Show relevant columns (Author/Title if available, otherwise all columns)
        display_cols = ["book_id"]
        for col in ["Author", "Title", "author", "title"]:
            if col in missing_meta_rows.columns:
                display_cols.append(col)
        if len(display_cols) > 1:
            display(missing_meta_rows[display_cols].head(20))
        else:
            display(missing_meta_rows.head(20))
        
        # Save missing books report
        missing_report_path = RESULTS_DIR / "stage10_correlation_analysis" / "missing_books_in_outputs.csv"
        missing_meta_rows.to_csv(missing_report_path, index=False)
        print(f"\n✓ Saved missing books report: {missing_report_path}")
    else:
        print("✅ All metadata books are present in outputs!")
else:
    print(f"⚠️ book_topic_probs.parquet not found at {book_topic_path}")
    missing_from_outputs = []

# --- Check chapters.csv coverage ---
if chapters_path.exists() and missing_from_outputs:
    print(f"\n{'='*60}")
    print("Checking chapters.csv coverage...")
    print(f"{'='*60}")
    
    chapters = pd.read_csv(chapters_path)
    
    # Try to detect id column
    chap_id_col = None
    for col in ["book_id", "goodreads_book_id", "ID", "id"]:
        if col in chapters.columns:
            chap_id_col = col
            break
    
    if chap_id_col is None:
        print(f"⚠️ chapters.csv has no obvious id col. Columns: {chapters.columns.tolist()}")
    else:
        chapters["book_id_norm"] = norm(chapters[chap_id_col])
        chap_set = set(chapters["book_id_norm"].dropna().unique())
        
        missing_in_chapters = sorted(set(missing_from_outputs) - chap_set)
        present_in_chapters = sorted(set(missing_from_outputs) & chap_set)
        
        print(f"📚 Missing in chapters.csv: {len(missing_in_chapters)}")
        if missing_in_chapters:
            print(f"   IDs: {missing_in_chapters}")
            print("   → These books were never scraped/extracted (upstream issue)")
        
        print(f"\n📚 Present in chapters.csv (but missing later): {len(present_in_chapters)}")
        if present_in_chapters:
            print(f"   IDs: {present_in_chapters}")
            print("   → These books were filtered out during preprocessing")
            
            # Show counts of chapters/text availability for books present in chapters
            tmp = chapters[chapters["book_id_norm"].isin(present_in_chapters)].copy()
            
            # Guess text column names
            text_col = None
            for col in ["text", "chapter_text", "Text", "Chapter Text"]:
                if col in tmp.columns:
                    text_col = col
                    break
            
            if text_col:
                tmp["text_len"] = tmp[text_col].astype("string").str.len()
                summary = tmp.groupby("book_id_norm").agg(
                    chapters=("book_id_norm", "size"),
                    avg_text_len=("text_len", "mean"),
                    min_text_len=("text_len", "min"),
                    max_text_len=("text_len", "max"),
                ).reset_index()
                
                # Merge with metadata to show titles (check for various column name variants)
                meta_cols = ["book_id"]
                for col_pair in [("Author", "Title"), ("author", "title")]:
                    if col_pair[0] in meta.columns and col_pair[1] in meta.columns:
                        meta_cols.extend(col_pair)
                        break
                
                if len(meta_cols) > 1:
                    summary = summary.merge(
                        meta[meta_cols],
                        left_on="book_id_norm",
                        right_on="book_id",
                        how="left"
                    )
                
                print(f"\n   Summary for {len(present_in_chapters)} books present in chapters.csv:")
                display(summary)
            else:
                print(f"   Found {len(tmp)} rows in chapters.csv for these books")
                print(f"   (Could not find text column to analyze content)")
else:
    if not chapters_path.exists():
        print(f"\n⚠️ chapters.csv not found at {chapters_path}")
    elif not missing_from_outputs:
        print(f"\n✅ No missing books to trace")

# --- Check sentence_df coverage ---
if sent_path.exists() and missing_from_outputs:
    print(f"\n{'='*60}")
    print("Checking sentence_df_with_topics.parquet coverage...")
    print(f"{'='*60}")
    
    # Only load book_id column to save memory
    sent = pd.read_parquet(sent_path, columns=["book_id"])
    sent["book_id_norm"] = norm(sent["book_id"])
    sent_set = set(sent["book_id_norm"].dropna().unique())
    
    missing_in_sentence_df = sorted(set(missing_from_outputs) - sent_set)
    present_in_sentence_df = sorted(set(missing_from_outputs) & sent_set)
    
    print(f"📄 Missing in sentence_df: {len(missing_in_sentence_df)}")
    if missing_in_sentence_df:
        print(f"   IDs: {missing_in_sentence_df}")
        print("   → These books were filtered out before sentence_df creation")
    
    print(f"\n📄 Present in sentence_df (but missing in outputs): {len(present_in_sentence_df)}")
    if present_in_sentence_df:
        print(f"   IDs: {present_in_sentence_df}")
        print("   → These books exist in sentence_df but were lost during aggregation")
        print("   ⚠️ This suggests an aggregation/filtering bug (less likely)")
        
        # Count sentences for these books
        sent_counts = sent[sent["book_id_norm"].isin(present_in_sentence_df)].groupby("book_id_norm").size().reset_index(name="sentence_count")
        print(f"\n   Sentence counts:")
        display(sent_counts)
else:
    if not sent_path.exists():
        print(f"\n⚠️ sentence_df_with_topics.parquet not found at {sent_path}")
    elif not missing_from_outputs:
        print(f"\n✅ No missing books to trace")

# --- Final summary ---
if missing_from_outputs:
    print(f"\n{'='*60}")
    print("DIAGNOSTIC SUMMARY")
    print(f"{'='*60}")
    print(f"Total missing books: {len(missing_from_outputs)}")
    if chapters_path.exists():
        print(f"  - Missing in chapters.csv: {len(missing_in_chapters) if 'missing_in_chapters' in locals() else 'N/A'}")
        print(f"  - Present in chapters.csv but missing later: {len(present_in_chapters) if 'present_in_chapters' in locals() else 'N/A'}")
    if sent_path.exists():
        print(f"  - Missing in sentence_df: {len(missing_in_sentence_df) if 'missing_in_sentence_df' in locals() else 'N/A'}")
        print(f"  - Present in sentence_df but missing in outputs: {len(present_in_sentence_df) if 'present_in_sentence_df' in locals() else 'N/A'}")


📊 Missing from outputs: 5 books
   Missing IDs: ['19561986', '19619918', '25781538', '52061964', '53491034']

📋 Metadata for missing books:


,book_id,Author,Title
38,53491034,s.l. scott,the billionaire's salvation: max
58,52061964,shain rose,reverie
70,25781538,melody anne,the tycoon's revenge
71,19561986,melody anne,the tycoon's vacation
72,19619918,melody anne,the tycoon's proposal



✓ Saved missing books report: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/missing_books_in_outputs.csv

Checking chapters.csv coverage...
⚠️ chapters.csv has no obvious id col. Columns: ['Author', 'Book Title', 'Chapter', 'Sentence']

Checking sentence_df_with_topics.parquet coverage...
📄 Missing in sentence_df: 5
   IDs: ['19561986', '19619918', '25781538', '52061964', '53491034']
   → These books were filtered out before sentence_df creation

📄 Present in sentence_df (but missing in outputs): 0

DIAGNOSTIC SUMMARY
Total missing books: 5
  - Missing in chapters.csv: N/A
  - Present in chapters.csv but missing later: N/A
  - Missing in sentence_df: 5
  - Present in sentence_df but missing in outputs: 0


In [12]:
EXCLUDED_BOOK_IDS = ['19561986','19619918','25781538','52061964','53491034']
meta = meta[~meta["book_id"].astype("string").isin(EXCLUDED_BOOK_IDS)].copy()
print("Meta cohort unique book_id:", meta["book_id"].nunique())

Meta cohort unique book_id: 92
